# Session 1 — Versioning and Tracking Machine Learning Models using MLflow

**Goal:** learn the core MLflow workflow — log a training run, compare runs, and
reload a saved model — using nothing but a local folder as the tracking store.

This is a first-exposure, from-scratch walkthrough. For a deeper comparison across
several model types on a real dataset (with a model registry step), see
`8. CodeAlongs/4. MLFlow/mlflow_experiment_tracking.ipynb`.

## Why bother tracking experiments?

Every model you train has three things worth remembering: **what you fed it**
(hyperparameters), **how well it did** (metrics), and **the model itself** (so you can
reload it later without retraining). Without a tool, this lives in scattered print
statements and file names like `model_v2_final_FINAL.pkl`. MLflow gives all three a
structured, queryable home.

## Prerequisites

```bash
poetry install   # mlflow is already a project dependency
```
No account or cloud service needed — MLflow can track runs to a local folder.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("mlflow version:", mlflow.__version__)

## Step 1 — Point MLflow at a tracking store

`set_tracking_uri` tells MLflow where to write run data. Pointing it at a local
folder (`./mlruns`) is the simplest possible setup — a real team would instead point
this at a shared MLflow tracking server so everyone's runs land in one place.

In [ ]:
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("session1-iris-classification")

print("Tracking URI:", mlflow.get_tracking_uri())

## Step 2 — Load data and define a run

In [ ]:
X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"train: {len(X_train)}, test: {len(X_test)}")

## Step 3 — Log a run

Everything inside `with mlflow.start_run():` belongs to one run: the parameters that
define it, the metrics it produced, and the model artifact itself.

In [ ]:
with mlflow.start_run(run_name="logreg_C1.0") as run:
    C = 1.0
    model = LogisticRegression(max_iter=200, C=C)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))

    mlflow.log_param("C", C)
    mlflow.log_param("max_iter", 200)
    mlflow.log_metric("test_accuracy", acc)
    mlflow.sklearn.log_model(model, artifact_path="model")

    print(f"run_id={run.info.run_id}  test_accuracy={acc:.4f}")

## Step 4 — Log a few more runs with different hyperparameters

The point of tracking is to make "which setting worked best?" a query, not a memory
exercise.

In [ ]:
for C in [0.01, 0.1, 1.0, 10.0]:
    with mlflow.start_run(run_name=f"logreg_C{C}"):
        model = LogisticRegression(max_iter=200, C=C)
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))

        mlflow.log_param("C", C)
        mlflow.log_param("max_iter", 200)
        mlflow.log_metric("test_accuracy", acc)
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"C={C:<6} test_accuracy={acc:.4f}")

## Step 5 — Compare runs programmatically

`mlflow.search_runs` returns every logged run as a DataFrame — sortable, filterable,
no scrolling through printed output required.

In [ ]:
experiment = mlflow.get_experiment_by_name("session1-iris-classification")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id],
                              order_by=["metrics.test_accuracy DESC"])

cols = ["run_id", "tags.mlflow.runName", "params.C", "metrics.test_accuracy"]
print(runs_df[cols].to_string(index=False))

## Step 6 — Reload the best model

This is the payoff: a downstream service only needs a `run_id` to load the exact
model that produced a given run's metrics — no training code required.

In [ ]:
best_run_id = runs_df.iloc[0]["run_id"]
best_acc = runs_df.iloc[0]["metrics.test_accuracy"]

loaded_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")
reloaded_acc = accuracy_score(y_test, loaded_model.predict(X_test))

print(f"Best run:        {best_run_id}")
print(f"Logged accuracy: {best_acc:.4f}")
print(f"Reloaded model accuracy matches: {abs(reloaded_acc - best_acc) < 1e-9}")

## What to try next

* Launch the MLflow UI to browse these runs visually:
  `mlflow ui --backend-store-uri mlruns` then open http://localhost:5000
* Add `mlflow.log_figure()` calls to attach plots (confusion matrix, ROC curve) to a
  run, as shown in the deeper CodeAlong notebook.
* Move on to Session 2 (DVC) to version the *data* a model was trained on, the other
  half of full reproducibility that MLflow alone doesn't cover.